# Day 91: DML in Practice: Binary Treatments, Instruments and Estimands

*Beginner to Ninja: Math for Causal Inference*

Double machine learning is a recipe, not one estimator: pick an orthogonal score for the estimand you want, plug in cross-fitted learners, and solve. With a binary treatment and heterogeneous effects, the partially linear model (PLR) and the interactive model (IRM, the AIPW score) answer different questions: PLR gives an overlap-weighted average effect, IRM the ATE. Instruments lead to the PLIV score.

**Learning outcomes**

- Show that PLR with a binary treatment and heterogeneous effects estimates a variance-weighted average effect.
- Implement DML for the ATE (IRM/AIPW) and the ATT with cross-fitted learners and propensity trimming.
- Write the orthogonal score for the partially linear IV model.
- Choose the score that matches the policy question and report diagnostics.

## Theory

### The DML recipe

1. Pick an estimand and an **orthogonal score** $\psi(W; \theta, \eta)$ identifying it.
2. Fit the nuisances $\eta$ with any learners, **cross-fitted** (Day 90).
3. Solve $E_n[\psi(W; \hat\theta, \hat\eta)] = 0$; the SE comes from $\hat\psi$ (Day 87).

### Three standard models

| Model | Score | Estimand |
|---|---|---|
| PLR | $(\tilde Y - \theta\tilde D)\tilde D$ | $\theta$; with heterogeneity, a variance-weighted average |
| IRM | $\mu_1 - \mu_0 + \frac{D(Y - \mu_1)}{e} - \frac{(1 - D)(Y - \mu_0)}{1 - e} - \theta$ | ATE |
| PLIV | $(\tilde Y - \theta\tilde D)\tilde Z$ | LATE-type effect with instrument $Z$ |

Here $\tilde Y = Y - E[Y \mid X]$, $\tilde D = D - E[D \mid X]$, $\tilde Z = Z - E[Z \mid X]$.

### PLR under heterogeneity

If $D$ is binary and $Y = \mu_0(X) + \tau(X)D + \varepsilon$, the PLR estimand is

$$
\theta_{PLR} = \frac{E\big[e(X)(1 - e(X))\,\tau(X)\big]}{E\big[e(X)(1 - e(X))\big]},
$$

a weighted average that emphasises units with propensity near one half: the overlap weights of Day 82. It is precise and stable but is not the ATE unless effects are constant or unrelated to $e(X)$.

### The ATT score

$$
\psi_{ATT} = \frac{D(Y - \mu_0(X))}{p} - \frac{e(X)(1 - D)(Y - \mu_0(X))}{p\,(1 - e(X))} - \frac{D}{p}\theta, \qquad p = P(D = 1),
$$

needs only $\mu_0$ and $e$, and tolerates propensities near 1 better than the ATE score.

### Trimming and diagnostics

Clip or trim $\hat e$ to $[\alpha, 1 - \alpha]$ (often $\alpha = 0.01$–$0.05$). Trimming observations changes the estimand to the ATE on the trimmed population, so report it. Always report nuisance prediction quality (out-of-fold $R^2$, log-loss) and the propensity distribution.

## Part 1 · Warm-up

### Exercise 1 · Which score?

A ministry asks: "What would the average effect be if every unemployed worker were enrolled?" Which score fits?

<details><summary>Show solution</summary>

**Step 1.** That is the ATE over the whole population.

**Answer:** the IRM (AIPW) score; PLR targets an overlap-weighted average instead.

</details>

*Your work:*

### Exercise 2 · PLR weights

Two equal-sized groups have propensities $0.5$ and $0.9$ and effects $2$ and $6$. Compute the ATE and the PLR estimand.

<details><summary>Show solution</summary>

**Step 1.** ATE $= (2 + 6)/2 = 4$.

**Step 2.** Weights $e(1 - e)$: $0.25$ and $0.09$.

**Step 3.** PLR: $(0.25\times2 + 0.09\times6)/0.34 = 1.04/0.34 = 3.06$.

**Answer:** ATE 4, PLR 3.06: PLR leans towards the group with better overlap.

</details>

*Your work:*

### Exercise 3 · PLIV by hand

After cross-fitted residualisation, $\sum\tilde Z\tilde Y = 12$ and $\sum\tilde Z\tilde D = 40$. Compute $\hat\theta$.

<details><summary>Show solution</summary>

**Step 1.** Solve $\sum(\tilde Y - \theta\tilde D)\tilde Z = 0$: $\hat\theta = 12/40 = 0.3$.

</details>

*Your work:*

### Exercise 4 · Trimming changes the estimand

Trimming drops units with $\hat e > 0.95$. Whose effect is now being estimated?

<details><summary>Show solution</summary>

**Step 1.** The ATE among units with $e(X) \le 0.95$: those for whom non-treatment is a realistic possibility.

**Answer:** report the trimmed share and describe the trimmed population.

</details>

*Your work:*

### Exercise 5 · Nuisances needed

Which nuisance functions does each of PLR, IRM-ATE, IRM-ATT and PLIV need?

<details><summary>Show solution</summary>

**Step 1.** PLR: $E[Y \mid X]$, $E[D \mid X]$.

**Step 2.** IRM-ATE: $\mu_0, \mu_1, e$. IRM-ATT: $\mu_0, e$.

**Step 3.** PLIV: $E[Y \mid X]$, $E[D \mid X]$, $E[Z \mid X]$.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · The PLR estimand under heterogeneity

Let $D$ be binary, $Y = \mu_0(X) + \tau(X)D + \varepsilon$ with $E[\varepsilon \mid D, X] = 0$. Show that $\theta$ solving $E[(\tilde Y - \theta\tilde D)\tilde D] = 0$ is $E[e(1 - e)\tau]/E[e(1 - e)]$.

<details><summary>Show solution</summary>

**Step 1.** $\ell(X) = \mu_0 + \tau e$, so $\tilde Y = \tau(D - e) + \varepsilon = \tau\tilde D + \varepsilon$.

**Step 2.** $E[\tilde Y\tilde D] = E[\tau\tilde D^2] = E[\tau E[\tilde D^2 \mid X]] = E[\tau e(1 - e)]$.

**Step 3.** $E[\tilde D^2] = E[e(1 - e)]$; divide. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · The PLIV score is orthogonal

With $\psi = (Y - \ell(X) - \theta(D - r(X)))(Z - q(X))$, $\ell = E[Y \mid X]$, $r = E[D \mid X]$, $q = E[Z \mid X]$ and the model $Y = \theta D + g(X) + \varepsilon$, $E[\varepsilon \mid Z, X] = 0$, show the derivatives in $\ell, r, q$ vanish.

<details><summary>Show solution</summary>

**Step 1.** In $\ell$ (direction $h$): $-E[h(X)(Z - q)] = 0$ since $E[Z - q \mid X] = 0$. Same for $r$: $\theta E[h(X)(Z - q)] = 0$.

**Step 2.** In $q$: $-E[(Y - \ell - \theta(D - r))h(X)]$. Since $\ell = \theta r + g$, the bracket is $\varepsilon$, and $E[\varepsilon h(X)] = 0$. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · The IRM score identifies the ATE

Show $E[\psi_{IRM}] = 0$ at the true nuisances iff $\theta = E[\mu_1(X) - \mu_0(X)]$.

<details><summary>Show solution</summary>

**Step 1.** $E[D(Y - \mu_1) \mid X] = e(X)\,E[Y - \mu_1 \mid D = 1, X] = 0$, and likewise for the control term.

**Step 2.** So $E[\psi] = E[\mu_1 - \mu_0] - \theta$. $\blacksquare$

</details>

*Your work:*

### Exercise 9 · The ATT score

Show $E[\psi_{ATT}] = 0$ iff $\theta = E[Y(1) - Y(0) \mid D = 1]$ under unconfoundedness.

<details><summary>Show solution</summary>

**Step 1.** $E[D(Y - \mu_0)]/p = E[Y(1) - \mu_0(X) \mid D = 1] = \text{ATT}$ (unconfoundedness gives $E[Y(0) \mid D = 1, X] = \mu_0(X)$).

**Step 2.** $E\big[\frac{e(1 - D)(Y - \mu_0)}{1 - e}\big] = E\big[e\,E[Y - \mu_0 \mid D = 0, X]\big] = 0$.

**Step 3.** $E[D\theta/p] = \theta$. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · Clipping and bias

Clipping replaces $\hat e$ by $\max(\hat e, \alpha)$ but keeps all units. Using Exercise 9 of Day 89, explain why clipping adds bias only where $e < \alpha$, and of what form.

<details><summary>Show solution</summary>

**Step 1.** The AIPW bias is $E[(e - \hat e)(\mu_1 - \hat\mu_1)/\hat e]$.

**Step 2.** Clipping changes $\hat e$ only where it was below $\alpha$, so only those units contribute extra bias, still as a product with the outcome-model error.

**Answer:** with a good outcome model the extra bias is small; clipping trades a little bias for a large variance reduction.

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · A job-training programme with heterogeneous effects

In a simulation with 3,000 workers, enrolment depends strongly on prior labour-market attachment, and earnings gains (in \$1,000s) are largest at the extremes of attachment, exactly where enrolment is nearly certain or nearly impossible. True ATE 0.99; PLR estimand 0.32. Compare naive, OLS, DML-PLR and DML-IRM.

<details><summary>Show solution</summary>

**Step 1.** Naive difference 1.93 and OLS with linear controls 0.63: both confounded or misspecified.

**Step 2.** DML-PLR: 0.33 (SE 0.06), on target for its estimand, the overlap-weighted effect.

**Step 3.** DML-IRM: 0.95 (SE 0.08), on target for the ATE.

**Answer:** both DML estimators work; they answer different questions, so choose the score from the policy question, not from the smaller SE.

</details>

*Your work:*

### Exercise 12 · Distance to college as an instrument

Card's study uses proximity to a college $Z$ as an instrument for schooling $D$, with many family controls. Describe the PLIV implementation.

<details><summary>Show solution</summary>

**Step 1.** Cross-fit learners for $E[Y \mid X]$, $E[D \mid X]$, $E[Z \mid X]$.

**Step 2.** $\hat\theta = \sum\tilde Z\tilde Y/\sum\tilde Z\tilde D$.

**Step 3.** Report the first-stage strength ($\sum\tilde Z\tilde D$ relative to its SE) to guard against weak instruments.

</details>

*Your work:*

### Exercise 13 · Hospital readmission programme: ATT

Hospitals chose to adopt a transitional-care programme; most large teaching hospitals adopted it ($\hat e$ up to 0.99). Why is the ATT a better target than the ATE here?

<details><summary>Show solution</summary>

**Step 1.** The ATE needs $1/(1 - e)$ weights for comparing adopters, and $1/e$ for non-adopters; with $e$ near 1, there are almost no comparable non-adopting teaching hospitals for the ATE.

**Step 2.** The ATT score only uses $e/(1 - e)$ for controls, and asks the natural question: what did the programme do for adopters?

</details>

*Your work:*

### Exercise 14 · Diagnostics table

List four diagnostics to report alongside a DML-IRM estimate of the effect of a smoking-cessation app.

<details><summary>Show solution</summary>

**Step 1.** Out-of-fold $R^2$ for the outcome models and log-loss/AUC for the propensity model.

**Step 2.** Distribution of $\hat e$ by treatment group and the share clipped or trimmed.

**Step 3.** Covariate balance after weighting.

**Step 4.** Sensitivity of the estimate to learner choice and to repeated splits.

</details>

*Your work:*

### Exercise 15 · Explaining two numbers

A report shows PLR 0.33 and IRM 0.95 for the same programme. Write a two-sentence explanation for a policy maker.

<details><summary>Show solution</summary>

**Step 1.** "The 0.95 figure is the average gain if every worker were enrolled; the 0.33 figure averages mainly over workers whose enrolment could plausibly have gone either way."

**Step 2.** "Gains are concentrated among workers who almost always or almost never enrol, so the two averages differ; neither is wrong."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold

rng = np.random.default_rng(91)
n = 3000; X = rng.normal(size=(n, 5))
e = 1 / (1 + np.exp(-(1.5 * X[:, 0] + .5 * X[:, 1] ** 2 - .5))); D = rng.binomial(1, e)
tau = 1 + 1.5 * (X[:, 0] ** 2 - 1); mu0 = np.sin(X[:, 0]) + X[:, 2] + .5 * X[:, 1] ** 2
Y = mu0 + tau * D + rng.normal(size=n)
w = e * (1 - e)
print(f"true ATE {tau.mean():.3f}   PLR estimand {(w * tau).sum() / w.sum():.3f}")
print(f"naive {Y[D == 1].mean() - Y[D == 0].mean():.3f}   OLS {np.linalg.lstsq(np.column_stack([np.ones(n), D, X]), Y, rcond=None)[0][1]:.3f}")

reg = lambda: RandomForestRegressor(200, min_samples_leaf=5, n_jobs=-1, random_state=0)
clf = lambda: RandomForestClassifier(200, min_samples_leaf=10, n_jobs=-1, random_state=0)
eh, m0, m1, lh = (np.empty(n) for _ in range(4))
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    eh[te] = clf().fit(X[tr], D[tr]).predict_proba(X[te])[:, 1]
    lh[te] = reg().fit(X[tr], Y[tr]).predict(X[te])
    t1, t0 = tr[D[tr] == 1], tr[D[tr] == 0]
    m1[te] = reg().fit(X[t1], Y[t1]).predict(X[te]); m0[te] = reg().fit(X[t0], Y[t0]).predict(X[te])

rD, rY = D - eh, Y - lh; th = rD @ rY / (rD @ rD); psi = (rY - th * rD) * rD
print(f"DML-PLR {th:.3f} (SE {np.sqrt(np.mean(psi ** 2) / np.mean(rD ** 2) ** 2 / n):.3f})")
ec = np.clip(eh, .02, .98)
phi = m1 - m0 + D * (Y - m1) / ec - (1 - D) * (Y - m0) / (1 - ec)
print(f"DML-IRM {phi.mean():.3f} (SE {phi.std() / np.sqrt(n):.3f})")

# Exercise 2: PLR weights by hand
e2, t2 = np.array([.5, .9]), np.array([2, 6]); w2 = e2 * (1 - e2)
print("Ex 2: ATE", t2.mean(), " PLR", round(float(w2 @ t2 / w2.sum()), 3))

## Key takeaways

- DML = orthogonal score for your estimand + cross-fitted learners + sandwich standard errors.
- With a binary treatment and heterogeneous effects, PLR estimates an overlap-weighted average; IRM (AIPW) estimates the ATE; the ATT score suits settings where treated units have propensities near 1.
- Trimming changes the estimand; report it along with nuisance accuracy and the propensity distribution.